In [ ]:
import os
import json
import torch
from tqdm import tqdm
from datasets import load_dataset

In [ ]:
print("Loading Fashionpedia...")
full_ds = load_dataset("detection-datasets/fashionpedia", split="train")


In [ ]:
# 2. Create 70/20/10 Splits
# First split off the test set (10%)
ds_30p = full_ds.train_test_split(test_size=0.7, seed=42)
ds_30p
test_ds = train_valid_test["test"]

In [ ]:
# Then split the remaining 90% into train (70% total) and valid (20% total)
    # 0.222 roughly equals 20% of the original 100%
    train_valid = train_valid_test["train"].train_test_split(test_size=0.222, seed=42)

In [ ]:
import os
import json
import torch
from tqdm import tqdm
from datasets import load_dataset

def prepare_fashionpedia_complete(output_dir="./fashionpedia_coco"):
    # 1. Load Dataset
    print("Loading Fashionpedia...")
    full_ds = load_dataset("detection-datasets/fashionpedia", split="train")

    # 2. Create 70/20/10 Splits
    # First split off the test set (10%)
    train_valid_test = full_ds.train_test_split(test_size=0.1, seed=42)
    test_ds = train_valid_test["test"]

    # Then split the remaining 90% into train (70% total) and valid (20% total)
    # 0.222 roughly equals 20% of the original 100%
    train_valid = train_valid_test["train"].train_test_split(test_size=0.222, seed=42)

    dataset_splits = {
        "train": train_valid["train"],
        "valid": train_valid["test"],
        "test": test_ds
    }

    for split, ds in dataset_splits.items():
        split_path = os.path.join(output_dir, split)
        os.makedirs(split_path, exist_ok=True)

        coco_data = {"images": [], "annotations": [], "categories": []}
        unique_cats = set()

        print(f"Processing {split} split ({len(ds)} images)...")
        for idx, sample in enumerate(tqdm(ds)):
            file_name = f"{idx}.jpg"
            img_path = os.path.join(split_path, file_name)
            sample['image'].save(img_path)

            coco_data["images"].append({
                "id": idx, "file_name": file_name,
                "width": sample['image'].size[0], "height": sample['image'].size[1]
            })

            obj_data = sample['objects']
            segs = obj_data.get('segmentation')
            for i in range(len(obj_data['category'])):
                bbox = obj_data['bbox'][i]
                coco_bbox = [float(bbox[1]), float(bbox[0]), float(bbox[3]-bbox[1]), float(bbox[2]-bbox[0])]
                seg = segs[i] if segs and i < len(segs) else []

                coco_data["annotations"].append({
                    "id": len(coco_data["annotations"]),
                    "image_id": idx,
                    "category_id": int(obj_data['category'][i]),
                    "bbox": coco_bbox,
                    "area": float(coco_bbox[2] * coco_bbox[3]),
                    "segmentation": seg,
                    "iscrowd": 0
                })
                unique_cats.add(obj_data['category'][i])

        for cat_id in sorted(list(unique_cats)):
            coco_data["categories"].append({
                "id": int(cat_id), "name": f"class_{cat_id}", "supercategory": "fashion"
            })

        with open(os.path.join(split_path, "_annotations.coco.json"), "w") as f:
            json.dump(coco_data, f)

# Execute
prepare_fashionpedia_complete()

In [ ]:
import os
import json
from tqdm import tqdm
from datasets import load_dataset

def prepare_fashionpedia_subset(
    output_dir="./fashionpedia_coco",
    subset_ratio=0.2,
    seed=42
):
    print("Loading Fashionpedia...")
    full_ds = load_dataset("detection-datasets/fashionpedia", split="train")

    # -------- STEP 1: sample 20% of dataset --------
    print(f"Sampling {int(subset_ratio*100)}% of dataset...")
    subset = full_ds.train_test_split(
        test_size=1 - subset_ratio,
        seed=seed
    )["train"]

    print(f"Subset size: {len(subset)} images")

    # -------- STEP 2: create 70/20/10 splits --------
    split_90_10 = subset.train_test_split(test_size=0.1, seed=seed)
    test_ds = split_90_10["test"]

    split_70_20 = split_90_10["train"].train_test_split(
        test_size=0.222,  # 20% of original
        seed=seed
    )

    dataset_splits = {
        "train": split_70_20["train"],
        "valid": split_70_20["test"],
        "test": test_ds
    }

    # -------- STEP 3: export to COCO --------
    for split, ds in dataset_splits.items():
        split_path = os.path.join(output_dir, split)
        os.makedirs(split_path, exist_ok=True)

        coco_data = {"images": [], "annotations": [], "categories": []}
        unique_cats = set()

        print(f"Processing {split} split ({len(ds)} images)...")

        for idx, sample in enumerate(tqdm(ds)):
            file_name = f"{idx}.jpg"
            img_path = os.path.join(split_path, file_name)
            sample["image"].save(img_path)

            coco_data["images"].append({
                "id": idx,
                "file_name": file_name,
                "width": sample["image"].size[0],
                "height": sample["image"].size[1],
            })

            obj_data = sample["objects"]
            segs = obj_data.get("segmentation")

            for i in range(len(obj_data["category"])):
                bbox = obj_data["bbox"][i]

                coco_bbox = [
                    float(bbox[1]),
                    float(bbox[0]),
                    float(bbox[3] - bbox[1]),
                    float(bbox[2] - bbox[0]),
                ]

                seg = segs[i] if segs and i < len(segs) else []

                coco_data["annotations"].append({
                    "id": len(coco_data["annotations"]),
                    "image_id": idx,
                    "category_id": int(obj_data["category"][i]),
                    "bbox": coco_bbox,
                    "area": float(coco_bbox[2] * coco_bbox[3]),
                    "segmentation": seg,
                    "iscrowd": 0,
                })

                unique_cats.add(obj_data["category"][i])

        for cat_id in sorted(unique_cats):
            coco_data["categories"].append({
                "id": int(cat_id),
                "name": f"class_{cat_id}",
                "supercategory": "fashion",
            })

        with open(os.path.join(split_path, "_annotations.coco.json"), "w") as f:
            json.dump(coco_data, f)

    print("\nDone!")

# Execute
prepare_fashionpedia_subset()

In [ ]:
import torch
from rfdetr import RFDETRNano

# Initialize model
model = RFDETRNano()


# Device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Train
model.train(
    # --- Data ---
    dataset_dir="../../data/fashionpedia_coco",
    output_dir="output",

    # --- Hardware ---
    device=device,
    batch_size=4,
    grad_accum_steps=4,   # effective batch = 16
    num_workers=2,

    # --- Training ---
    epochs=100,
    lr=5e-5,              # better for smaller datasets
    weight_decay=1e-4,
    use_ema=True,
    early_stopping=True,
    early_stopping_patience=10,

    # --- Accuracy ---
    resolution=512,       # safer; use 640 if VRAM allows
    multi_scale=True,

    # --- Logging ---
    checkpoint_interval=5,
    tensorboard=True
)

In [4]:
from inference import GarmentDetector

model = GarmentDetector("output/checkpoint_best_regular.pth")

[2026-03-01 18:20:56] [INFO] rf-detr - File rf-detr-nano.pth already exists with correct MD5 hash.


[2026-03-01 18:20:56] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-03-01 18:20:56] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-03-01 18:20:56] [INFO] rf-detr - Loading pretrain weights


[2026-03-01 18:20:57] [WARNING] rf-detr - Reinitializing detection head with 90 classes based on pretrained weights, configured for 46.


In [5]:
from PIL import Image

model.predict(Image.open("../../examples/ex5.jpeg"))

[2026-03-01 18:20:57] [WARNING] rf-detr - Model is not optimized for inference. Latency may be higher than expected. You can optimize the model for inference by calling model.optimize_for_inference().


[]